# Chunking Strategy Optimization Experiments

Based on RAG Paper: "Searching for Best Practices in Retrieval-Augmented Generation"

This notebook tests 5 different chunking methods and measures their impact on RAG performance using TRACe metrics.

## Step 1: Install Dependencies

In [1]:
!pip install -q python-dotenv datasets tiktoken langchain-core langchain-text-splitters langchain-huggingface langchain-chroma langchain-openai nltk


[notice] A new release of pip is available: 25.2 -> 26.2
[notice] To update, run: pip install --upgrade pip


## Step 2: Import Libraries

In [2]:
import os
import json
import re
import numpy as np
import pandas as pd
import tiktoken
import shutil
from dotenv import load_dotenv
from datasets import load_dataset
from nltk.tokenize import sent_tokenize
import nltk

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

nltk.download('punkt_tab', quiet=True)

# Load environment
load_dotenv()
openrouter_token = os.environ.get('OPENROUTER_TOKEN')

print("✓ All imports successful")

/Users/saikrishna/Desktop/Codespace/IIIT_AI_ML_course/Capstone_Project/reliablerag/.venv-1/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ All imports successful


## Step 3: Load Dataset

In [ ]:
import sys, os


def _add_ragbench_lib_to_path():
    for candidate in (os.getcwd(), os.path.join(os.getcwd(), "delucion_dataset")):
        if os.path.isdir(os.path.join(candidate, "ragbench_lib")) and candidate not in sys.path:
            sys.path.insert(0, candidate)
            return


_add_ragbench_lib_to_path()

from ragbench_lib.data_loading import load_rag_bench_data

print("Loading dataset...")
DATASET_NAME = "delucionqa"  # feeds both load_rag_bench_data() and vector store naming below
docs_df = load_rag_bench_data(DATASET_NAME, num_samples=50)
print(f"✓ Loaded {len(docs_df)} documents from {docs_df['row_id'].nunique()} unique questions")


## Step 4: Setup Embedding Model & LLM

In [ ]:
from ragbench_lib.models import get_embedding_model, get_generation_llm, get_judge_llm
from ragbench_lib.generation_prompt import RAG_GENERATION_PROMPT
from ragbench_lib.vector_store import get_persist_dir, vector_store_names

# Embedding model
embedding_model = get_embedding_model(openrouter_token)

# LLM for generation
llm = get_generation_llm(openrouter_token)

# LLM for annotation (using 70B for better quality)
llama_judge = get_judge_llm(openrouter_token)

# Prompt template
prompt = RAG_GENERATION_PROMPT

print("✓ Embedding model and LLM configured")


## Step 5: Define Chunking Strategies

In [5]:
encoding = tiktoken.get_encoding("cl100k_base")

def count_tokens(text):
    return len(encoding.encode(text))

def chunk_by_tokens(text, target_tokens, overlap_tokens):
    """Split text to achieve target token count per chunk"""
    tokens = encoding.encode(text)
    chunks = []
    
    i = 0
    while i < len(tokens):
        chunk_tokens = tokens[i : i + target_tokens]
        chunk_text = encoding.decode(chunk_tokens)
        chunks.append(chunk_text)
        i += target_tokens - overlap_tokens
    
    return chunks

def chunking_method_1_baseline(docs_df):
    """M1: Baseline - 128 tokens (small baseline)"""
    chunks = []
    for _, doc in docs_df.iterrows():
        splits = chunk_by_tokens(doc["text"], target_tokens=128, overlap_tokens=16)
        for chunk_idx, chunk_text in enumerate(splits):
            chunks.append({
                "method": "M1_Baseline_128tokens",
                "chunk_id": f"{doc['doc_id']}_c{chunk_idx}",
                "text": chunk_text,
                "tokens": count_tokens(chunk_text),
                "row_id": doc['row_id'],
            })
    return chunks

def chunking_method_2_optimized_size(docs_df):
    """M2: Optimized - 256 token chunks (no overlap)"""
    chunks = []
    for _, doc in docs_df.iterrows():
        splits = chunk_by_tokens(doc["text"], target_tokens=256, overlap_tokens=0)
        for chunk_idx, chunk_text in enumerate(splits):
            chunks.append({
                "method": "M2_Optimized_256tokens",
                "chunk_id": f"{doc['doc_id']}_c{chunk_idx}",
                "text": chunk_text,
                "tokens": count_tokens(chunk_text),
                "row_id": doc['row_id'],
            })
    return chunks

def chunking_method_3_sliding_window(docs_df):
    """M3: Sliding Window - 512 tokens with 50% overlap (PAPER WINNER)"""
    chunks = []
    for _, doc in docs_df.iterrows():
        splits = chunk_by_tokens(doc["text"], target_tokens=512, overlap_tokens=256)
        for chunk_idx, chunk_text in enumerate(splits):
            chunks.append({
                "method": "M3_SlidingWindow_512tokens",
                "chunk_id": f"{doc['doc_id']}_c{chunk_idx}",
                "text": chunk_text,
                "tokens": count_tokens(chunk_text),
                "row_id": doc['row_id'],
            })
    return chunks

def chunking_method_4_small_to_big(docs_df):
    """M4: Small-to-Big - 256 token chunks"""
    chunks = []
    for _, doc in docs_df.iterrows():
        splits = chunk_by_tokens(doc["text"], target_tokens=256, overlap_tokens=32)
        for chunk_idx, chunk_text in enumerate(splits):
            chunks.append({
                "method": "M4_SmallToBig_256tokens",
                "chunk_id": f"{doc['doc_id']}_c{chunk_idx}",
                "text": chunk_text,
                "tokens": count_tokens(chunk_text),
                "row_id": doc['row_id'],
            })
    return chunks

def chunking_method_5_with_metadata(docs_df):
    """M5: Optimized + Metadata - 512 tokens with 25% overlap"""
    chunks = []
    for _, doc in docs_df.iterrows():
        splits = chunk_by_tokens(doc["text"], target_tokens=512, overlap_tokens=128)
        for chunk_idx, chunk_text in enumerate(splits):
            chunks.append({
                "method": "M5_SlidingWindow_WithMetadata",
                "chunk_id": f"{doc['doc_id']}_c{chunk_idx}",
                "text": chunk_text,
                "tokens": count_tokens(chunk_text),
                "row_id": doc['row_id'],
            })
    return chunks

print("✓ Chunking methods defined (now using token-based splitting)")

✓ Chunking methods defined (now using token-based splitting)


## Step 6: Chunking Statistics Comparison

In [6]:
def compare_chunking_methods(docs_df):
    """Compare statistics of all 5 chunking methods"""
    print("\n" + "="*100)
    print("CHUNKING STRATEGY COMPARISON")
    print("="*100 + "\n")
    
    methods = [
        ("M1 (Baseline)", chunking_method_1_baseline),
        ("M2 (Optimized Size)", chunking_method_2_optimized_size),
        ("M3 (Sliding Window)", chunking_method_3_sliding_window),
        ("M4 (Small-to-Big)", chunking_method_4_small_to_big),
        ("M5 (With Metadata)", chunking_method_5_with_metadata),
    ]
    
    stats = []
    for method_name, method_func in methods:
        print(f"Processing {method_name}...", end=" ")
        chunks = method_func(docs_df)
        
        tokens = [c["tokens"] for c in chunks]
        
        stats.append({
            "Method": method_name,
            "Total Chunks": len(chunks),
            "Avg Tokens": f"{np.mean(tokens):.0f}",
            "Min Tokens": min(tokens),
            "Max Tokens": max(tokens),
            "Std Dev": f"{np.std(tokens):.0f}",
        })
        print(f"✓ ({len(chunks)} chunks)")
    
    df = pd.DataFrame(stats)
    print("\n" + "="*100)
    print("STATISTICS")
    print("="*100)
    display(df)
    
    print("\n" + "-"*100)
    print("Paper Recommendations:")
    print("  • Optimal chunk size: 512 tokens (M2, M3, M5)")
    print("  • Best technique: Sliding window (M3)")
    print("  • Expected improvement: +7-13% in Context Relevance")
    print("-"*100)

compare_chunking_methods(docs_df)


CHUNKING STRATEGY COMPARISON

Processing M1 (Baseline)... ✓ (450 chunks)
Processing M2 (Optimized Size)... ✓ (239 chunks)
Processing M3 (Sliding Window)... ✓ (239 chunks)
Processing M4 (Small-to-Big)... ✓ (260 chunks)
Processing M5 (With Metadata)... ✓ (190 chunks)

STATISTICS


,Method,Total Chunks,Avg Tokens,Min Tokens,Max Tokens,Std Dev
0,M1 (Baseline),450,104,1,128,39
1,M2 (Optimized Size),239,175,4,256,82
2,M3 (Sliding Window),239,230,4,512,154
3,M4 (Small-to-Big),260,174,1,256,83
4,M5 (With Metadata),190,241,5,512,152



----------------------------------------------------------------------------------------------------
Paper Recommendations:
  • Optimal chunk size: 512 tokens (M2, M3, M5)
  • Best technique: Sliding window (M3)
  • Expected improvement: +7-13% in Context Relevance
----------------------------------------------------------------------------------------------------


## Step 7: TRACe Metrics Functions

In [7]:
from ragbench_lib.chunking import get_sentences
from ragbench_lib.trace_eval import (
    format_documents_with_keys,
    annotate_response_for_metrics as _annotate_response_for_metrics,
    compute_context_relevance,
    compute_utilization,
    compute_completeness,
    compute_adherence,
)


def annotate_response_for_metrics(documents, question, response):
    """Annotate a response using this notebook's judge LLM (llama_judge)."""
    return _annotate_response_for_metrics(llama_judge, documents, question, response)


print("✓ TRACe metric functions ready (ragbench_lib.trace_eval)")

✓ TRACe metric functions ready (ragbench_lib.trace_eval)


## Step 8: Vector Store & RAG Setup

In [ ]:
import tempfile

def build_rag_pipeline(method_name, docs_df, embedding_model, llm, prompt):
    """Build complete RAG pipeline with selected chunking method"""
    
    methods = {
        'M1': chunking_method_1_baseline,
        'M2': chunking_method_2_optimized_size,
        'M3': chunking_method_3_sliding_window,
        'M4': chunking_method_4_small_to_big,
        'M5': chunking_method_5_with_metadata,
    }
    
    if method_name not in methods:
        raise ValueError(f"Invalid method. Choose from: {list(methods.keys())}")
    
    print(f"\nBuilding RAG pipeline with {method_name}...")
    
    # Get chunks
    print("  Creating chunks...", end=" ")
    chunks = methods[method_name](docs_df)
    print(f"✓ ({len(chunks)} chunks)")
    
    # Convert to documents
    documents = [Document(page_content=c["text"], metadata={"chunk_id": c["chunk_id"], "row_id": c["row_id"]}) for c in chunks]
    
    # Use temporary directory for each experiment to avoid database locks
    print("  Building vector store...", end=" ")
    prefix, collection_name = vector_store_names(DATASET_NAME, method_name)
    persist_dir = get_persist_dir(prefix)
    
    try:
        vector_store = Chroma.from_documents(
            documents=documents,
            embedding=embedding_model,
            collection_name=collection_name,
            persist_directory=persist_dir,
        )
    except Exception as e:
        shutil.rmtree(persist_dir, ignore_errors=True)
        raise e
    
    print("✓")
    
    # Create retriever
    retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 8})
    
    # Create RAG chain
    def format_docs(docs):
        return "\n\n".join(doc.page_content for doc in docs)
    
    rag_chain = ({"context": retriever | format_docs, "question": RunnablePassthrough()} | prompt | llm | StrOutputParser())
    
    return rag_chain, retriever, vector_store

print("✓ RAG pipeline builder defined")

## Step 9: Experiment Runner

In [9]:
def run_experiment(method_name, docs_df, embedding_model, llm, prompt, num_samples=10):
    """Run complete experiment with selected chunking method"""
    
    print(f"\n{'='*80}")
    print(f"EXPERIMENT: {method_name} ({num_samples} samples)")
    print('='*80 + "\n")
    
    # Build pipeline
    rag_chain, retriever, _ = build_rag_pipeline(method_name, docs_df, embedding_model, llm, prompt)
    
    # Run evaluation
    unique_samples = docs_df.drop_duplicates(subset=['row_id']).head(num_samples).reset_index(drop=True)
    results = []
    
    print("Running evaluation...")
    for i, row in unique_samples.iterrows():
        try:
            question = row['question']
            my_response = rag_chain.invoke(question)
            retrieved_docs = retriever.invoke(question)
            retrieved_texts = [doc.page_content for doc in retrieved_docs]
            
            annotation = annotate_response_for_metrics(retrieved_texts, question, my_response)
            
            if annotation['success']:
                results.append({
                    'context_relevance': compute_context_relevance(retrieved_texts, annotation),
                    'utilization': compute_utilization(retrieved_texts, annotation),
                    'completeness': compute_completeness(annotation),
                    'adherence': compute_adherence(annotation),
                })
                print(f"  [{i+1}/{num_samples}] ✓")
            else:
                print(f"  [{i+1}/{num_samples}] ✗ Annotation failed")
        except Exception as e:
            print(f"  [{i+1}/{num_samples}] ✗ Error: {str(e)[:50]}")
    
    # Aggregate
    if results:
        avg_metrics = {
            'context_relevance': np.mean([r['context_relevance'] for r in results]),
            'utilization': np.mean([r['utilization'] for r in results]),
            'completeness': np.mean([r['completeness'] for r in results]),
            'adherence': np.mean([r['adherence'] for r in results]),
        }
        
        print(f"\n{'='*80}")
        print(f"RESULTS: {method_name}")
        print('='*80)
        print(f"  Context Relevance: {avg_metrics['context_relevance']:.4f}")
        print(f"  Utilization:       {avg_metrics['utilization']:.4f}")
        print(f"  Completeness:      {avg_metrics['completeness']:.4f}")
        print(f"  Adherence:         {avg_metrics['adherence']:.4f}")
        print('='*80 + "\n")
        
        return avg_metrics
    else:
        print("No successful evaluations!")
        return None

print("✓ Experiment runner defined")

✓ Experiment runner defined


## Step 10: Run Experiments

### Quick Test (10 samples each)

In [10]:
# Test M1 (Baseline)
m1_results = run_experiment('M1', docs_df, embedding_model, llm, prompt, num_samples=10)


EXPERIMENT: M1 (10 samples)


Building RAG pipeline with M1...
  Creating chunks... ✓ (450 chunks)
  Building vector store... ✓
Running evaluation...
  [1/10] ✓
  [2/10] ✓
  [3/10] ✓
  [4/10] ✓
  [5/10] ✓
  [6/10] ✓
  [7/10] ✗ Annotation failed
  [8/10] ✓
  [9/10] ✓
  [10/10] ✓

RESULTS: M1
  Context Relevance: 0.1280
  Utilization:       0.0521
  Completeness:      0.5031
  Adherence:         0.8889



In [11]:
# Test M3 (Sliding Window - Paper Winner)
m3_results = run_experiment('M3', docs_df, embedding_model, llm, prompt, num_samples=10)


EXPERIMENT: M3 (10 samples)


Building RAG pipeline with M3...
  Creating chunks... ✓ (239 chunks)
  Building vector store... ✓
Running evaluation...
  [1/10] ✓
  [2/10] ✗ Annotation failed
  [3/10] ✓
  [4/10] ✓
  [5/10] ✓
  [6/10] ✓
  [7/10] ✓
  [8/10] ✓
  [9/10] ✓
  [10/10] ✓

RESULTS: M3
  Context Relevance: 0.0472
  Utilization:       0.0164
  Completeness:      0.4241
  Adherence:         0.8889



### Compare Results

In [12]:
if m1_results and m3_results:
    print("\n" + "="*100)
    print("COMPARISON: M1 vs M3")
    print("="*100 + "\n")
    
    comparison = pd.DataFrame({
        'M1 (Baseline)': m1_results,
        'M3 (Sliding Window)': m3_results
    }).T
    
    display(comparison)
    
    # Calculate improvements
    print("\nImprovement (M3 vs M1):")
    for metric in ['context_relevance', 'utilization', 'completeness', 'adherence']:
        m1 = m1_results[metric]
        m3 = m3_results[metric]
        improve = ((m3 - m1) / m1 * 100) if m1 > 0 else 0
        print(f"  {metric}: {m1:.4f} → {m3:.4f} ({improve:+.1f}%)")



COMPARISON: M1 vs M3



,context_relevance,utilization,completeness,adherence
M1 (Baseline),0.127956,0.052078,0.503078,0.888889
M3 (Sliding Window),0.047178,0.016378,0.424067,0.888889



Improvement (M3 vs M1):
  context_relevance: 0.1280 → 0.0472 (-63.1%)
  utilization: 0.0521 → 0.0164 (-68.6%)
  completeness: 0.5031 → 0.4241 (-15.7%)
  adherence: 0.8889 → 0.8889 (+0.0%)


### Test All 5 Methods (Optional - Uncomment to run)

In [14]:
all_experiments = {}
# 
for method in ['M1', 'M2', 'M3', 'M4', 'M5']:
    results = run_experiment(method, docs_df, embedding_model, llm, prompt, num_samples=10)
    all_experiments[method] = results
# 
# Create comparison table
comparison_table = pd.DataFrame(all_experiments).T
print("\n" + "="*100)
print("FINAL COMPARISON: ALL METHODS")
print("="*100)
display(comparison_table)
# 
# Find winner
best_method = comparison_table['context_relevance'].idxmax()
print(f"\n🏆 Best Method: {best_method}")
print(f"Context Relevance: {comparison_table.loc[best_method, 'context_relevance']:.4f}")


EXPERIMENT: M1 (10 samples)


Building RAG pipeline with M1...
  Creating chunks... ✓ (450 chunks)
  Building vector store... ✓
Running evaluation...
  [1/10] ✓
  [2/10] ✓
  [3/10] ✓
  [4/10] ✓
  [5/10] ✓
  [6/10] ✗ Annotation failed
  [7/10] ✓
  [8/10] ✓
  [9/10] ✓
  [10/10] ✓

RESULTS: M1
  Context Relevance: 0.1446
  Utilization:       0.0523
  Completeness:      0.4083
  Adherence:         0.8889


EXPERIMENT: M2 (10 samples)


Building RAG pipeline with M2...
  Creating chunks... ✓ (239 chunks)
  Building vector store... ✓
Running evaluation...
  [1/10] ✓
  [2/10] ✓
  [3/10] ✓
  [4/10] ✓
  [5/10] ✓
  [6/10] ✓
  [7/10] ✓
  [8/10] ✓
  [9/10] ✓
  [10/10] ✓

RESULTS: M2
  Context Relevance: 0.0716
  Utilization:       0.0386
  Completeness:      0.6048
  Adherence:         1.0000


EXPERIMENT: M3 (10 samples)


Building RAG pipeline with M3...
  Creating chunks... ✓ (239 chunks)
  Building vector store... ✓
Running evaluation...
  [1/10] ✓
  [2/10] ✓
  [3/10] ✓
  [4/10] ✓
  [5/10] ✓


,context_relevance,utilization,completeness,adherence
M1,0.144622,0.052256,0.408322,0.888889
M2,0.071640,0.038570,0.604760,1.000000
M3,0.054478,0.026411,0.507400,0.888889
M4,0.075180,0.032630,0.517490,0.900000
M5,0.072056,0.030622,0.494300,1.000000



🏆 Best Method: M1
Context Relevance: 0.1446
